# Experimento — TabKAN

TabKAN (Eslamian et al., 2025): KAN para tabela. Reimplementação própria com 3 famílias de base (spline, Chebyshev, Fourier); Jacobi, Padé, fracionária, FastKAN e mixer não foram implementados.

Protocolo comum da pré-proposta: mesmas 5 dobras e mesmo teste de `if702-preprocessamento.ipynb` (via `comum.py`), KS como métrica e critério da busca, 8 hiperparâmetros e o mesmo N de trials no Optuna (`TPESampler(seed=42)`, `MedianPruner`, trial 0 = padrão, estudo em SQLite). Entropia cruzada e MSE entram como um dos 8 hiperparâmetros (`perda`).

In [1]:
import json, numpy as np, pandas as pd
import optuna, comum, modelos_jeff as M
folds, final, colunas = comum.carregar_dobras()   # CSV: variável de ambiente CHURN_CSV ou dados/telco_ibm.csv

/usr/local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Estudo Optuna

Para refazer a busca (leva de dezenas de minutos a horas; é retomável): `st = M.estudo(MODELO, N)`. Aqui só lemos o estudo salvo.

In [2]:
MODELO = 'tabkan'
st = optuna.load_study(study_name=MODELO, storage=f'sqlite:///resultados/{MODELO}.db')
df = st.trials_dataframe()[['number','state','value']+[c for c in st.trials_dataframe().columns if c.startswith('params_')]]
df.sort_values('value', ascending=False).head(10)

,number,state,value,params_base_cheb,params_base_fourier,params_base_spline,params_familia,params_l2,params_larg,params_lr_adam,params_lr_lbfgs,params_otim,params_perda,params_prof
20,20,PRUNED,0.563257,NaN,NaN,3.0,spline,0.006001,10,NaN,0.168797,lbfgs,ce,4
18,18,PRUNED,0.562736,NaN,NaN,5.0,spline,0.004179,25,0.003691,NaN,adam,mse,3
14,14,PRUNED,0.560228,NaN,NaN,5.0,spline,0.002222,25,NaN,0.294677,lbfgs,ce,1
28,28,PRUNED,0.559980,NaN,NaN,5.0,spline,0.008769,20,NaN,0.128843,lbfgs,ce,3
26,26,PRUNED,0.556233,NaN,NaN,9.0,spline,0.009228,20,NaN,0.214659,lbfgs,ce,2
17,17,PRUNED,0.550707,NaN,NaN,5.0,spline,0.000158,30,NaN,0.122753,lbfgs,ce,2
12,12,COMPLETE,0.543192,NaN,NaN,5.0,spline,0.003480,25,NaN,0.157534,lbfgs,ce,2
30,30,COMPLETE,0.540274,NaN,NaN,5.0,spline,0.008705,25,NaN,0.204774,lbfgs,mse,1
2,2,COMPLETE,0.539809,NaN,NaN,5.0,spline,0.001383,25,NaN,0.119966,lbfgs,mse,2
9,9,COMPLETE,0.539809,NaN,NaN,5.0,spline,0.001383,25,NaN,0.119966,lbfgs,mse,2


## 2. Resultados finais (`resultados/<modelo>.json`)

In [3]:
r = json.load(open(f'resultados/{MODELO}.json'))
print('Melhor configuração:', r['melhor_config'])
e = r['estudo']
print(f"KS trial 0 = {e['ks_trial0']:.4f} -> melhor trial = {e['melhor_ks']:.4f} | completos {e['n_completos']}, podados {e['n_podados']}")
print('Importâncias (Optuna):', {k: round(v,3) for k,v in e['importancias'].items()})
pd.DataFrame(r['ce_vs_mse']['teste']).T[['ks','auroc','ce','mse','ce_corr','f1','precision','recall']]

Melhor configuração: {'familia': 'spline', 'prof': 2, 'larg': 25, 'base': 5, 'otim': 'lbfgs', 'lr': 0.1575338772621255, 'l2': 0.003479722935938827, 'perda': 'ce'}
KS trial 0 = 0.4762 -> melhor trial = 0.5432 | completos 15, podados 15
Importâncias (Optuna): {'larg': 0.471, 'lr_lbfgs': 0.256, 'l2': 0.23, 'base_spline': 0.017, 'prof': 0.012, 'otim': 0.006, 'familia': 0.005, 'perda': 0.003, 'base_fourier': 0.0, 'lr_adam': 0.0, 'base_cheb': 0.0}


,ks,auroc,ce,mse,ce_corr,f1,precision,recall
ce,0.546912,0.848691,0.470964,0.157121,0.413391,0.63587,0.55102,0.751606
mse,0.542501,0.845651,0.472527,0.156985,0.419192,0.631141,0.516349,0.811563


## 3. Entropia cruzada × MSE (mesma configuração, mesmas dobras e sementes)

In [4]:
c = r['ce_vs_mse']
print(f"ΔKS validação (CE−MSE) = {c['delta_ks_val_media']:+.4f} ± {c['delta_ks_val_dp']:.4f} | CE melhor em {c['dobras_ce_melhor']}/5 dobras | Wilcoxon p = {c['wilcoxon_p']:.3f}")
print(f"ΔKS teste (CE−MSE) = {c['delta_ks_teste']:+.4f}  IC95% {np.round(c['delta_ks_teste_ic95'],4)}")
pd.DataFrame({k: {m: v[m][0] for m in v} for k, v in c['resumo'].items()})

ΔKS validação (CE−MSE) = -0.0036 ± 0.0102 | CE melhor em 1/5 dobras | Wilcoxon p = 0.125
ΔKS teste (CE−MSE) = +0.0044  IC95% [-0.006   0.0168]


,ce,mse
ks,0.535333,0.538924
auroc,0.841899,0.841149
ce_raw,0.479887,0.478288
mse_raw,0.159782,0.158505
ce_corr,0.421884,0.424313
mse_corr,0.136539,0.137538
tr_ce,0.474760,0.485483
tr_mse,0.155636,0.158983


## 4. Limitações desta rodada

- Rodada em CPU, N = 30 trials; versão final pretendida: GPU T4 (`colab_stab_tabkan.ipynb`), N comum do grupo.
- Early stopping usa a mesma dobra que escolhe a configuração (KS de validação levemente otimista).
- O KS do teste é de um comitê das 5 dobras × 5 sementes, avaliado uma única vez.